In [ ]:
import numpy as np
import pandas as pd

from itertools import combinations
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold


def ranking_spearman(ranking1, ranking2):
    """
    Compute Spearman correlation between two feature rankings.

    Parameters
    ----------
    ranking1, ranking2 : list
        Ordered feature names, from most to least important.

    Returns
    -------
    rho : float
        Spearman rank correlation.
    """

    # Use common features only
    common_features = list(set(ranking1) & set(ranking2))

    if len(common_features) < 2:
        return np.nan

    # Convert feature names to numeric rank positions
    rank1 = {feature: i + 1 for i, feature in enumerate(ranking1)}
    rank2 = {feature: i + 1 for i, feature in enumerate(ranking2)}

    r1 = [rank1[feature] for feature in common_features]
    r2 = [rank2[feature] for feature in common_features]

    rho, _ = spearmanr(r1, r2)

    return rho


def calculate_stability(
    X,
    y,
    ranking_function,
    n_folds=5,
    random_state=42,
    **ranking_kwargs
):
    """
    Calculate feature-ranking stability for any FSA.

    Procedure
    ---------
    1. Divide the dataset into three approximately equal subsets.
    2. Apply 5-fold stratified cross-validation to each subset.
    3. Generate one feature ranking for each fold.
    4. This produces 3 x 5 = 15 rankings.
    5. Compute Spearman correlation between every pair of rankings.
    6. Average all pairwise correlations to obtain Stability_raw.
    7. Normalize Stability_raw from [-1, 1] to [0, 1].

    Parameters
    ----------
    X : pandas.DataFrame
        Feature matrix.

    y : pandas.Series
        Target variable.

    ranking_function : callable
        Function that receives (X_train, y_train) and returns
        an ordered list of feature names from most to least important.

    n_folds : int
        Number of CV folds used in each subset.

    random_state : int
        Random seed for reproducibility.

    **ranking_kwargs :
        Additional parameters required by the ranking function.

    Returns
    -------
    stability_raw : float
        Mean pairwise Spearman correlation in [-1, 1].

    stability_normalized : float
        Normalized stability score in [0, 1].

    rankings : list
        All generated feature rankings.

    correlations : list
        All pairwise Spearman correlations.
    """

    # ---------------------------------------------------------
    # 1. Divide dataset into three approximately equal subsets
    # ---------------------------------------------------------

    X_temp, X_part3, y_temp, y_part3 = train_test_split(
        X,
        y,
        test_size=1 / 3,
        stratify=y,
        random_state=random_state
    )

    X_part1, X_part2, y_part1, y_part2 = train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        stratify=y_temp,
        random_state=random_state
    )

    subsets = [
        (X_part1, y_part1),
        (X_part2, y_part2),
        (X_part3, y_part3)
    ]

    rankings = []

    # ---------------------------------------------------------
    # 2. Apply 5-fold CV inside each subset
    # ---------------------------------------------------------

    for X_subset, y_subset in subsets:

        cv = StratifiedKFold(
            n_splits=n_folds,
            shuffle=True,
            random_state=random_state
        )

        for train_idx, _ in cv.split(X_subset, y_subset):

            X_fold_train = X_subset.iloc[train_idx]
            y_fold_train = y_subset.iloc[train_idx]

            # -------------------------------------------------
            # 3. Generate one ranking using the selected FSA
            # -------------------------------------------------

            ranking = ranking_function(
                X_fold_train,
                y_fold_train,
                **ranking_kwargs
            )

            rankings.append(list(ranking))

    # ---------------------------------------------------------
    # 4. Calculate Spearman correlation for every ranking pair
    # ---------------------------------------------------------

    correlations = []

    for ranking1, ranking2 in combinations(rankings, 2):

        rho = ranking_spearman(
            ranking1,
            ranking2
        )

        if not np.isnan(rho):
            correlations.append(rho)

    if len(correlations) == 0:
        raise ValueError(
            "No valid pairwise Spearman correlations could be calculated."
        )

    # ---------------------------------------------------------
    # 5. Raw Stability = average pairwise Spearman correlation
    # ---------------------------------------------------------

    stability_raw = np.mean(correlations)

    # ---------------------------------------------------------
    # 6. Normalize Stability from [-1, 1] to [0, 1]
    # ---------------------------------------------------------

    stability_normalized = (stability_raw + 1) / 2

    return (
        stability_raw,
        stability_normalized,
        rankings,
        correlations
    )

In [ ]:
# Chi-square as FSA example 

In [ ]:
from sklearn.feature_selection import chi2


def chi_square_ranking(X_train, y_train):

    scores, _ = chi2(X_train, y_train)

    ranking = (
        pd.Series(
            scores,
            index=X_train.columns
        )
        .fillna(0.0)
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    return ranking

In [ ]:
stability_raw, stability, rankings, correlations = calculate_stability(
    X,
    y,
    ranking_function=chi_square_ranking
)

print(f"Raw Stability: {stability_raw:.4f}")
print(f"Normalized Stability: {stability:.4f}")
print(f"Number of generated rankings: {len(rankings)}")
print(f"Number of pairwise comparisons: {len(correlations)}")